# Sistema Agéntico para Análisis Predictivo — Premier League

**Proyecto Final — Programación para Analítica de Datos 2026-I**

Este notebook ejecuta el pipeline agéntico completo mediante el agente orquestador,
que coordina las tres skills de forma secuencial:

1. **Skill 1** — Preparación de datos (`skill_preparacion.py`)
2. **Skill 2** — Análisis exploratorio EDA (`skill_eda.py`)
3. **Skill 3** — Modelado predictivo (`skill_modelado.py`)

---

## 0. Configuración del entorno

In [ ]:
# Aseguramos que el directorio raíz del proyecto esté en el path
import sys
import os

# Subir un nivel desde notebooks/ hasta la raíz del proyecto
raiz_proyecto = os.path.abspath(os.path.join(os.getcwd(), '..'))
if raiz_proyecto not in sys.path:
    sys.path.insert(0, raiz_proyecto)

os.chdir(raiz_proyecto)
print(f'Directorio de trabajo: {os.getcwd()}')
print(f'Python path incluye: {raiz_proyecto}')

In [ ]:
# Verificar que la base de datos existe
DB_PATH = 'data/premier_league.db'

if os.path.exists(DB_PATH):
    print(f'Base de datos encontrada: {DB_PATH}')
else:
    print(f'ADVERTENCIA: No se encontró {DB_PATH}')
    print('Asegúrate de que el archivo SQLite existe antes de continuar.')

---
## 1. Ejecución del Pipeline Completo (vía Orquestador)

El agente orquestador coordina las tres skills. Solo necesitamos llamar a `ejecutar_pipeline()`.

In [ ]:
from scripts.orchestrator import ejecutar_pipeline

# Ejecutar el pipeline agéntico completo
resumen = ejecutar_pipeline(DB_PATH)

---
## 2. Resumen Ejecutivo de Resultados

In [ ]:
print('=' * 60)
print('RESUMEN EJECUTIVO DEL PIPELINE AGÉNTICO')
print('=' * 60)
print(f"\nModelo seleccionado : {resumen['modelo_ganador']}")
print(f"Accuracy Test       : {resumen['accuracy_test']:.4f}")
print(f"F1 Macro            : {resumen['f1_macro']:.4f}")
print(f"CV Mean (5-fold)    : {resumen['cv_mean']:.4f}")
print(f"\nConclusión EDA:")
print(resumen['eda_conclusion'])
print(f"\nConclusión Modelado:")
print(resumen['modelo_conclusion'])

---
## 3. Visualización de Gráficas Generadas

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

graficas = resumen['graficas_generadas']
graficas_existentes = [g for g in graficas if os.path.exists(g)]

print(f'Gráficas disponibles: {len(graficas_existentes)}/{len(graficas)}')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
titulos = [
    '1. Distribución de Resultados',
    '2. Promedio Goles por Equipo',
    '3. Heatmap de Correlación',
    '4. Comparación de Modelos',
    '5. Matriz de Confusión',
    '6. Importancia de Features'
]

for i, (ax, titulo) in enumerate(zip(axes.flatten(), titulos)):
    if i < len(graficas_existentes):
        img = mpimg.imread(graficas_existentes[i])
        ax.imshow(img)
        ax.set_title(titulo, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Gráficas del Pipeline Agéntico — Premier League', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Ejecución Individual de Skills (opcional)

Las skills también pueden invocarse de forma individual para inspección.

In [ ]:
# Skill 1 individual
from scripts.skills.skill_preparacion import preparar_datos

df = preparar_datos(DB_PATH)
print(f'\nDataset preparado: {df.shape}')
df.head()

In [ ]:
# Skill 2 individual
from scripts.skills.skill_eda import ejecutar_eda

reporte_eda = ejecutar_eda(df)
print('\nClaves del reporte EDA:', list(reporte_eda.keys()))

In [ ]:
# Skill 3 individual
from scripts.skills.skill_modelado import ejecutar_modelado

reporte_modelado = ejecutar_modelado(df)
print('\nModelo ganador:', reporte_modelado['mejor_modelo']['nombre'])
print('Accuracy Test: ', reporte_modelado['mejor_modelo']['acc_test'])

---
## 5. Arquitectura del Sistema

```
ejecutar_pipeline()          ← AGENTE ORQUESTADOR
    │
    ├── preparar_datos()     ← SKILL 1: Carga, limpieza, feature engineering
    │       └── retorna df (DataFrame enriquecido)
    │
    ├── ejecutar_eda(df)     ← SKILL 2: EDA + 3 gráficas
    │       └── retorna reporte_eda (Dict)
    │
    ├── ejecutar_modelado(df) ← SKILL 3: ML + selección + 3 gráficas
    │       └── retorna reporte_modelado (Dict)
    │
    └── consolidar_resultados() → resumen ejecutivo final
```

**Principio de diseño:** El orquestador solo coordina. Toda la lógica analítica vive en las skills.